recuperons les données depuis load_breast_cancer. Enregistrons les futures et les targets respectivement dans les dataframe et series X et y.

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dimensions :", X.shape)
print("\nRépartition des classes :")
print(y.value_counts())

Dimensions : (569, 30)

Répartition des classes :
target
1    357
0    212
Name: count, dtype: int64


separons les donées de teste des données d'entrainement tout en s'assurant de la bonne distribution des classe entre les 2.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y          #stratify to maintain class distribution
)

verifions la distribution des classes entre l'ensemble d'entrement et celui de test

In [6]:
print(y_train.value_counts())
print(y_test.value_counts())

target
1    285
0    170
Name: count, dtype: int64
target
1    72
0    42
Name: count, dtype: int64


importons et configurons les elements necessaires a la validation croisée

In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


effectuer la cross validation

In [ ]:

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("Scores :", scores)
print("Moyenne :", scores.mean())
print("Écart-type :", scores.std())

Scores : [0.96703297 0.98901099 0.97802198 0.98901099 0.96703297]
Moyenne : 0.9780219780219781
Écart-type : 0.009828870230768321


Nous allons maintenant comparer plusieurs modele et comprendre pourquoi on ne se limite pas à un seul algorithme.

In [11]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    ),

    "SVM": make_pipeline(
        StandardScaler(),
        SVC()
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    )
}

comparer les modeles 


In [12]:
results = {}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    results[name] = {
        "mean": scores.mean(),
        "std": scores.std()
    }

In [13]:
print(results)

{'Logistic Regression': {'mean': np.float64(0.9780219780219781), 'std': np.float64(0.009828870230768321)}, 'SVM': {'mean': np.float64(0.9692307692307693), 'std': np.float64(0.014578570507056717)}, 'Random Forest': {'mean': np.float64(0.9626373626373628), 'std': np.float64(0.01785502946073838)}}


creation d'un pipeline standardiser pour svm et definition d'une grille de parametre a tester

In [ ]:
svm_pipeline = make_pipeline(
    StandardScaler(),
    SVC()
)

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"]
}